*** Peptide Library Designer *** 
Turn a limited set of peptide sequences into focused combinatorial libraries.

This self-contained notebook pools residues at corresponding positions and uses integer linear programming to choose libraries under explicit synthesis limits. It reports input coverage, complete library membership, and the scope of the optimization certificate. No local Python modules are needed.

Set INPUT_PATH to your sequence file. Residue choices, peptide lengths, and fixed or variable positions are derived from that file. No peptide templates, predefined residue pools, or example-data generator are embedded in the implementation.

Run the cells from top to bottom. The main settings are in section 2.


In [23]:
from pathlib import Path
from dataclasses import dataclass
from itertools import product
from collections import defaultdict
import csv
import hashlib
import io
import json
import math
import platform
import re
import time
import warnings
import numpy as np
import pandas as pd
import scipy
from scipy.optimize import milp, Bounds, LinearConstraint
from scipy.sparse import csr_matrix, hstack, vstack, eye
import matplotlib.pyplot as plt

2. Settings
MAX_LIBRARIES is the total across all length/group partitions. MAX_LIBRARY_SIZE
is the number of peptide combinations in each library. No optimization uses predicted
affinity implicitly: set WEIGHT_COLUMN only if you have a justified, nonnegative priority
score. Duplicate sequences within a group are counted once, retaining the maximum weight.

Equal lengths are treated as already positionally comparable. If peptides use different
scaffolds or binding registers, provide a GROUP_COLUMN to prevent mixing them. This tool
does not perform alignment or hierarchical clustering.

In auto mode the code tries exhaustive candidate generation within bounded resources.
If a bound is reached, it uses a restricted candidate pool plus multi-start greedy searches.
An optimal solver result then applies only to that pool. Use exact to require exhaustive
generation (it raises an error if the budget is reached), or heuristic to skip enumeration.

In [ ]:
INPUT_PATH = Path("peptides.csv")    # Set this to your CSV/TSV/FASTA/plain-text file
OUTPUT_DIR = Path("outputs/run")    # Re-running overwrites files in this directory

RUN_LABEL = "Peptide library design" # Optional descriptive title for figures and metadata
SEQUENCE_COLUMN = None               # Auto-detect sequence, peptide, seq, peptide_sequence, binder_sequence
ID_COLUMN = None                     # Auto-detect id or peptide_id; otherwise generate IDs
WEIGHT_COLUMN = None                 # None = equal weight; extra CSV columns are preserved as metadata
GROUP_COLUMN = None                  # Optional scaffold/family/target column; lengths are always separated
TOKEN_MODE = "letters"              # letters, braces ("{A}{NMe-Phe}"), or space ("A NMe-Phe")
INVALID_SEQUENCE_POLICY = "skip"    # skip invalid records with a report, or error to stop

MAX_LIBRARIES = 4
MAX_LIBRARY_SIZE = 150
CANDIDATE_MODE = "auto"             # auto, exact, heuristic
MAX_CANDIDATES_PER_GROUP = 12000
MAX_ENUMERATION_TRANSITIONS = 250000
GREEDY_RESTARTS = 6
RANDOM_SEED = 7
SOLVER_TIME_LIMIT = 60               # seconds PER optimization stage; up to two stages

LOGO_MODE = "probability"           # probability (frequency) or information (bits)
LOGO_POSITIONS_PER_ROW = 30          # Wrap long peptides across multiple rows

3. Read and normalize sequences
Supported inputs are CSV/TSV, FASTA, or plain text with one sequence per line.
A .log filename works only if its contents follow one of these formats; arbitrary
log messages are not parsed as peptides. In letter mode, unknown symbols such as X
and gaps are excluded with a report by default. Set INVALID_SEQUENCE_POLICY = "error"
to stop instead. Unknown symbols are never replaced, expanded, or treated as concrete
amino acids. Extra CSV columns are retained in input_records.csv; they affect the
optimization only when explicitly selected as weights or groups. FASTA headers provide
identifiers and wrapped sequence lines are joined. Noncanonical monomers in explicit
token modes are treated as distinct tokens with no assumed
chemical equivalence. Terminal modifications must be handled consistently outside
the variable sequence or explicitly encoded as tokens.

In [25]:
AMINO_ACIDS = set('ACDEFGHIKLMNPQRSTVWY')

def read_sequences(path):
    """CSV/TSV, FASTA, or one sequence per line; never scrape arbitrary log prose."""
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f'Input not found: {path}. Set INPUT_PATH to an existing file.')
    raw = path.read_text(encoding='utf-8-sig')
    lines = [s.strip() for s in raw.splitlines() if s.strip()]
    if not lines:
        raise ValueError('The input file is empty.')
    if lines[0].startswith('>'):
        rows, current_id, seq = [], None, []
        for line in lines:
            if line.startswith('>'):
                if current_id is not None:
                    rows.append({'id': current_id, 'sequence': ''.join(seq)})
                current_id, seq = line[1:].strip(), []
                if not current_id:
                    raise ValueError('Every FASTA header must contain an identifier.')
            else:
                seq.append(line)
        rows.append({'id': current_id, 'sequence': ''.join(seq)})
        return pd.DataFrame(rows)
    headers = {'sequence', 'peptide', 'seq', 'peptide_sequence', 'binder_sequence'}
    if (path.suffix.lower() not in {'.csv', '.tsv'}
            and lines[0].lower() not in headers
            and all(re.fullmatch(r'[A-Za-z]+', s) for s in lines)):
        return pd.DataFrame({'sequence': lines})
    try:
        delimiter = csv.Sniffer().sniff('\n'.join(lines[:30]), delimiters=',\t;').delimiter
    except csv.Error:
        delimiter = '\t' if path.suffix.lower() == '.tsv' else ','
    frame = pd.read_csv(io.StringIO(raw), sep=delimiter, dtype=str, keep_default_na=False)
    frame.columns = [str(c).strip() for c in frame.columns]
    return frame


def tokenize(sequence, mode='letters'):
    text = str(sequence).strip()
    if mode == 'letters':
        tokens = tuple(re.sub(r'\s+', '', text).upper())
        invalid = set(tokens) - AMINO_ACIDS
        if invalid:
            raise ValueError(f'Noncanonical or unknown amino-acid symbols: {sorted(invalid)}. '
                             'Unknown residues are not substituted or treated as concrete monomers.')
    elif mode == 'braces':
        compact = re.sub(r'\}\s*\{', '}{', text)
        tokens = tuple(t.strip() for t in re.findall(r'\{([^{}]+)\}', compact))
        if not tokens or ''.join('{' + t + '}' for t in tokens) != compact:
            raise ValueError('Brace sequences must be complete {monomer}{monomer} strings.')
    elif mode == 'space':
        tokens = tuple(text.split())
    else:
        raise ValueError('TOKEN_MODE must be letters, braces, or space.')
    if not tokens or any(not t for t in tokens):
        raise ValueError('Empty peptide sequence.')
    return tokens


def sequence_text(tokens, mode):
    if mode == 'letters':
        return ''.join(tokens)
    if mode == 'braces':
        return ''.join('{' + t + '}' for t in tokens)
    return ' '.join(tokens)


def prepare_input(frame, sequence_column=None, id_column=None, weight_column=None,
                  group_column=None, token_mode='letters', invalid_policy='skip'):
    """Validate sequence records, retaining original row IDs and an explicit exclusion table."""
    if invalid_policy not in {'skip', 'error'}:
        raise ValueError('INVALID_SEQUENCE_POLICY must be skip or error.')
    if token_mode not in {'letters', 'braces', 'space'}:
        raise ValueError('TOKEN_MODE must be letters, braces, or space.')
    if frame.empty:
        raise ValueError('No input records were found.')
    lower = {str(c).lower(): c for c in frame.columns}
    if sequence_column is None:
        choices = [lower[k] for k in ['sequence','peptide','seq','peptide_sequence','binder_sequence'] if k in lower]
        if len(choices) != 1:
            raise ValueError(f'Set SEQUENCE_COLUMN explicitly; available columns: {list(frame.columns)}')
        sequence_column = choices[0]
    if sequence_column not in frame:
        raise ValueError(f'Column not found: {sequence_column}')
    source = frame.copy()
    id_column = id_column or lower.get('id') or lower.get('peptide_id')
    if id_column is None:
        id_column = '__input_record_id'
        while id_column in source.columns:
            id_column += '_'
        source[id_column] = [f'peptide_{i:03d}' for i in range(1, len(source)+1)]
    elif id_column not in source:
        raise ValueError(f'Column not found: {id_column}')
    ids = source[id_column]
    if ids.isna().any() or (ids.astype(str).str.strip() == '').any() or ids.astype(str).str.strip().duplicated().any():
        raise ValueError('Input identifiers must be nonempty and unique, including excluded rows.')
    accepted, excluded = [], []
    for offset, (_, row) in enumerate(source.iterrows(), 1):
        try:
            if pd.isna(row[sequence_column]):
                raise ValueError('Missing sequence.')
            tokenize(row[sequence_column], token_mode)
        except ValueError as exc:
            if invalid_policy == 'error':
                raise ValueError(f'Input record {offset} ({row[id_column]}): {exc}') from exc
            excluded.append({'input_record': offset, 'id': row[id_column],
                             'sequence': row[sequence_column], 'reason': str(exc)})
        else:
            accepted.append(offset-1)
    exclusions = pd.DataFrame(excluded, columns=['input_record', 'id', 'sequence', 'reason'])
    if not accepted:
        raise ValueError('No valid sequences remain. First exclusion: ' + exclusions.iloc[0]['reason'])
    if excluded:
        warnings.warn(f'Excluded {len(excluded)} of {len(source)} input records with invalid sequences. '
                      'See the printed exclusion table and excluded_sequences.csv. '
                      'Set INVALID_SEQUENCE_POLICY="error" to stop instead.')
    parents, normalized = normalize_input(source.iloc[accepted], sequence_column, id_column,
                                          weight_column, group_column, token_mode)
    return parents, normalized, exclusions


def normalize_input(frame, sequence_column=None, id_column=None, weight_column=None,
                    group_column=None, token_mode='letters'):
    if frame.empty:
        raise ValueError('No input records were found.')
    if len(set(frame.columns)) != len(frame.columns):
        raise ValueError('Duplicate column names are not supported.')
    lower = {str(c).lower(): c for c in frame.columns}
    if sequence_column is None:
        choices = [lower[k] for k in ['sequence','peptide','seq','peptide_sequence','binder_sequence'] if k in lower]
        if len(choices) != 1:
            raise ValueError(f'Set SEQUENCE_COLUMN explicitly; available columns: {list(frame.columns)}')
        sequence_column = choices[0]
    if id_column is None:
        id_column = lower.get('id') or lower.get('peptide_id')
    for c in [sequence_column, id_column, weight_column, group_column]:
        if c is not None and c not in frame:
            raise ValueError(f'Column not found: {c}')
    originals, merged, seen_ids = [], {}, set()
    for offset, (_, row) in enumerate(frame.iterrows(), 1):
        if pd.isna(row[sequence_column]):
            raise ValueError(f'Missing sequence in input row {offset}.')
        tokens = tokenize(row[sequence_column], token_mode)
        label = f'peptide_{offset:03d}' if id_column is None else str(row[id_column]).strip()
        if not label or label in seen_ids or (id_column and pd.isna(row[id_column])):
            raise ValueError(f'Empty or duplicate identifier in row {offset}: {label!r}')
        seen_ids.add(label)
        group = str(row[group_column]).strip() if group_column else 'all'
        if not group or (group_column and pd.isna(row[group_column])):
            raise ValueError(f'Missing group label in row {offset}.')
        weight = 1.0 if weight_column is None else float(row[weight_column])
        if not np.isfinite(weight) or weight < 0:
            raise ValueError('Weights must be finite and nonnegative; transform negative scores explicitly.')
        key = (group, len(tokens))
        normalized = sequence_text(tokens, token_mode)
        originals.append({'id': label, 'sequence': normalized, 'group': group,
                          'length': len(tokens), 'weight': weight})
        dedup_key = (key, tokens)
        if dedup_key not in merged:
            merged[dedup_key] = {'tokens': tokens, 'group_key': key, 'ids': [], 'weight': weight}
        merged[dedup_key]['ids'].append(label)
        merged[dedup_key]['weight'] = max(merged[dedup_key]['weight'], weight)
    parents = list(merged.values())
    if not any(p['weight'] > 0 for p in parents):
        raise ValueError('At least one input sequence must have positive weight.')
    for i, parent in enumerate(parents):
        parent['index'] = i
    return parents, pd.DataFrame(originals)

4. Generate candidate libraries
Each library is a Cartesian product of position-specific residue sets. Its size is
the product of the numbers of allowed residues. A sequence is covered only when its
residue is present at every position and its length/group matches. Common residues
remain fixed automatically. Only monomers present at that same position in the input
are used; the tool does not introduce chemical analogues from an external inventory.

In [26]:
@dataclass(frozen=True)
class Candidate:
    group: tuple
    options: tuple
    size: int
    covered: tuple


def box_for(tokens):
    return tuple((token,) for token in tokens)


def union_box(box, tokens):
    return tuple(tuple(sorted(set(options) | {token})) for options, token in zip(box, tokens))


def box_size(box):
    return math.prod(len(options) for options in box)


def covered_parents(box, indices, parents):
    return tuple(i for i in indices if all(t in options for t, options in zip(parents[i]['tokens'], box)))


def build_candidates(parents, max_size=150, mode='auto', max_candidates_per_group=12000,
                     transition_limit=250000, greedy_restarts=6, seed=7):
    """Exact bounded enumeration; auto completes a truncated pool with multi-start greedy searches."""
    if mode not in {'auto', 'exact', 'heuristic'}:
        raise ValueError('CANDIDATE_MODE must be auto, exact, or heuristic.')
    limits = [max_size, max_candidates_per_group, transition_limit, greedy_restarts]
    if any(not isinstance(v, (int, np.integer)) or isinstance(v, bool) or v < 1 for v in limits):
        raise ValueError('Candidate limits must be positive integers.')
    groups = defaultdict(list)
    for p in parents:
        groups[p['group_key']].append(p['index'])
    candidates, reports = [], []
    rng = np.random.default_rng(seed)
    for group, indices in sorted(groups.items()):
        if len(indices) > max_candidates_per_group:
            raise ValueError('Candidate budget must accommodate all singleton libraries.')
        states = set()
        transitions, complete, stop_reason = 0, mode != 'heuristic', None
        if mode != 'heuristic':
            for i in indices:
                before = sorted(states)
                states.add(box_for(parents[i]['tokens']))
                for old in before:
                    transitions += 1
                    new = union_box(old, parents[i]['tokens'])
                    if box_size(new) <= max_size:
                        states.add(new)
                    if len(states) >= max_candidates_per_group or transitions >= transition_limit:
                        complete = False
                        stop_reason = 'candidate budget' if len(states) >= max_candidates_per_group else 'transition budget'
                        break
                if not complete:
                    break
            if not complete and mode == 'exact':
                raise RuntimeError(f'Exact enumeration reached the {stop_reason} for group {group}. '
                                   'Increase the budget or choose auto/heuristic mode.')
        # Reserve space for multi-start candidates if exhaustive enumeration stopped.
        singleton_boxes = {box_for(parents[i]['tokens']) for i in indices}
        if not complete:
            reserve = max(len(indices), max_candidates_per_group // 2)
            old = sorted(states, key=lambda b: (box_size(b), b))[:max(0, reserve-len(singleton_boxes))]
            states = set(old) | singleton_boxes
            for restart in range(greedy_restarts):
                for anchor in rng.permutation(indices):
                    box = box_for(parents[int(anchor)]['tokens'])
                    while len(states) < max_candidates_per_group:
                        covered = set(covered_parents(box, indices, parents))
                        proposals = []
                        for j in indices:
                            if j in covered:
                                continue
                            new = union_box(box, parents[j]['tokens'])
                            size = box_size(new)
                            if size > max_size:
                                continue
                            cov = covered_parents(new, indices, parents)
                            gain = sum(parents[t]['weight'] for t in cov if t not in covered)
                            gain_count = len(set(cov) - covered)
                            cost = max(1, size - box_size(box))
                            if restart % 3 == 0:
                                score = (gain/cost, gain_count/cost, -size)
                            elif restart % 3 == 1:
                                score = (gain, gain_count, -size)
                            else:
                                score = (float(rng.random()), gain, -size)
                            proposals.append((score, new))
                        if not proposals:
                            break
                        box = max(proposals, key=lambda x: (x[0], x[1]))[1]
                        states.add(box)
        else:
            states |= singleton_boxes
        # Equivalent coverage: retain the smallest library; valid for the coverage and cost objectives.
        best = {}
        for box in sorted(states):
            cov = covered_parents(box, indices, parents)
            cand = Candidate(group, box, box_size(box), cov)
            if cov not in best or (cand.size, cand.options) < (best[cov].size, best[cov].options):
                best[cov] = cand
        candidates.extend(sorted(best.values(), key=lambda c: (c.size, c.options)))
        reports.append({'group': group[0], 'length': group[1], 'parents': len(indices),
                        'enumeration_complete': complete, 'stop_reason': stop_reason,
                        'raw_candidates': len(states), 'retained_candidates': len(best),
                        'transitions': transitions,
                        'method': 'exhaustive' if complete else 'bounded candidates + multi-start greedy'})
    return candidates, reports

5. Optimize selection
Binary variables select libraries and count covered input sequences. First maximize
weighted unique coverage under the library-count limit. If that objective is
certified optimal, a second solve minimizes the sum of selected library sizes while
preserving coverage within numerical tolerance. Overlapping peptides are counted in
each library for synthesis cost. Solver timeouts are reported; they do not become
optimality claims.

In [27]:
def optimize_libraries(parents, candidates, max_libraries=4, time_limit=60):
    """Maximize weighted unique-parent coverage, then minimize summed library size."""
    if not isinstance(max_libraries, (int, np.integer)) or max_libraries < 1 or time_limit <= 0:
        raise ValueError('MAX_LIBRARIES must be a positive integer and TIME_LIMIT must be positive.')
    m, n = len(candidates), len(parents)
    if not m:
        raise ValueError('No feasible candidate libraries.')
    rows, cols = [], []
    for j, c in enumerate(candidates):
        rows.extend(c.covered)
        cols.extend([j] * len(c.covered))
    incidence = csr_matrix((np.ones(len(rows)), (rows, cols)), shape=(n, m))
    # x_j: choose candidate j. y_i: count parent i as covered (once).
    coverage = hstack([-incidence, eye(n)], format='csr')
    budget = csr_matrix(np.r_[np.ones(m), np.zeros(n)].reshape(1, -1))
    matrix = vstack([coverage, budget], format='csr')
    constraints = [LinearConstraint(matrix, -np.inf, np.r_[np.zeros(n), max_libraries])]
    weights = np.array([p['weight'] for p in parents], dtype=float)
    scaled_weights = weights / weights.max()
    objective = np.r_[np.zeros(m), -scaled_weights]
    options = {'time_limit': float(time_limit), 'mip_rel_gap': 0.0, 'presolve': True}
    bounds = Bounds(np.zeros(m+n), np.ones(m+n))
    integrality = np.ones(m+n)

    def valid_result(result):
        if result.x is None or not np.all(np.isfinite(result.x)):
            return False
        if np.max(np.abs(result.x - np.round(result.x))) > 1e-5:
            return False
        x = np.round(result.x)
        return bool(np.all((x >= 0) & (x <= 1)) and np.all(matrix @ x <= np.r_[np.zeros(n), max_libraries] + 1e-6))

    primary = milp(objective, integrality=integrality, bounds=bounds,
                   constraints=constraints, options=options)
    if not valid_result(primary):
        raise RuntimeError(f'No validated feasible solution: {primary.message}. Increase TIME_LIMIT.')
    chosen = np.flatnonzero(primary.x[:m] > .5).tolist()
    if not chosen:
        raise RuntimeError('The solver did not select a library. Increase TIME_LIMIT to obtain a useful design.')
    initial_covered = set().union(*(set(candidates[j].covered) for j in chosen)) if chosen else set()
    achieved = float(sum(scaled_weights[i] for i in initial_covered))
    secondary = None
    # Tie-break only after primary optimality. Preserve coverage to numerical tolerance.
    if primary.status == 0:
        weighted_row = csr_matrix(np.r_[np.zeros(m), scaled_weights].reshape(1,-1))
        tolerance = 1e-8 * max(1.0, achieved)
        lock = LinearConstraint(weighted_row, achieved - tolerance, np.inf)
        sizes = np.array([c.size for c in candidates], dtype=float)
        secondary = milp(np.r_[sizes, np.zeros(n)], integrality=integrality, bounds=bounds,
                         constraints=constraints + [lock], options=options)
        if valid_result(secondary):
            alternative = np.flatnonzero(secondary.x[:m] > .5).tolist()
            cov = set().union(*(set(candidates[j].covered) for j in alternative)) if alternative else set()
            alternative_score = sum(scaled_weights[i] for i in cov)
            if (alternative_score >= achieved - tolerance
                    and sum(candidates[j].size for j in alternative) <= sum(candidates[j].size for j in chosen)):
                chosen = alternative
    selected = sorted([candidates[j] for j in chosen], key=lambda c: (c.group, c.options))
    covered = set().union(*(set(c.covered) for c in selected)) if selected else set()
    gap = getattr(primary, 'mip_gap', None)
    report = {'primary_status': int(primary.status), 'primary_message': primary.message,
              'primary_optimal_over_candidates': primary.status == 0,
              'primary_mip_gap': float(gap) if gap is not None and np.isfinite(gap) else None,
              'cost_status': int(secondary.status) if secondary is not None else None,
              'cost_message': secondary.message if secondary is not None else 'Skipped: primary optimum not certified.',
              'covered_unique_parents': len(covered), 'total_unique_parents': n,
              'weighted_coverage': float(sum(weights[i] for i in covered)),
              'total_parent_weight': float(weights.sum()),
              'selected_libraries': len(selected), 'summed_library_members': sum(c.size for c in selected)}
    if primary.status != 0:
        warnings.warn('Solver time limit reached: feasible solution returned, but optimality is not certified.')
    return selected, report

6. Export and verify results
Export CSV tables for designs, positional compositions, every library member, and
covered/uncovered inputs. run_metadata.json records input SHA-256, parameters,
versions, candidate-search completeness, and solver status. Every selected library
is checked against independently enumerated members and starting-sequence coverage.

In [28]:
def result_tables(parents, selected, token_mode='letters'):
    designs, compositions, members, cover_rows = [], [], [], []
    parent_by_group = defaultdict(dict)
    for p in parents:
        parent_by_group[p['group_key']][p['tokens']] = p
    library_ids = [f'LIB{i+1:02d}' for i in range(len(selected))]
    for library_id, cand in zip(library_ids, selected):
        ids = [label for i in cand.covered for label in parents[i]['ids']]
        designs.append({'library_id': library_id, 'group': cand.group[0], 'length': cand.group[1],
                        'size': cand.size, 'unique_parents_covered': len(cand.covered),
                        'parent_ids_json': json.dumps(ids),
                        'options_json': json.dumps(cand.options)})
        for position, options in enumerate(cand.options, 1):
            compositions.append({'library_id': library_id, 'position': position,
                                 'residue_options_json': json.dumps(options), 'option_count': len(options),
                                 'fixed_within_library': len(options) == 1})
        for number, tokens in enumerate(product(*cand.options), 1):
            parent = parent_by_group[cand.group].get(tokens)
            members.append({'library_id': library_id, 'member_number': number,
                            'group': cand.group[0], 'sequence': sequence_text(tokens, token_mode),
                            'is_input_sequence': parent is not None,
                            'parent_ids_json': json.dumps(parent['ids'] if parent else [])})
    for p in parents:
        libs = [lid for lid, c in zip(library_ids, selected) if p['index'] in c.covered]
        cover_rows.append({'parent_index': p['index'], 'parent_ids_json': json.dumps(p['ids']),
                           'sequence': sequence_text(p['tokens'], token_mode),
                           'group': p['group_key'][0], 'length': len(p['tokens']), 'weight': p['weight'],
                           'covered': bool(libs), 'libraries_json': json.dumps(libs)})
    return {'design': pd.DataFrame(designs), 'composition': pd.DataFrame(compositions),
            'members': pd.DataFrame(members), 'coverage': pd.DataFrame(cover_rows)}


def verify_design(parents, selected, tables, max_libraries, max_size):
    if len(selected) > max_libraries:
        raise AssertionError('Library-count limit violated.')
    for lid, cand in zip(tables['design']['library_id'], selected):
        if box_size(cand.options) != cand.size or cand.size > max_size:
            raise AssertionError('Library-size verification failed.')
        subset = tables['members'].query('library_id == @lid')
        if len(subset) != cand.size or subset['sequence'].nunique() != cand.size:
            raise AssertionError('Member enumeration is incomplete or contains duplicates.')
        expected = tuple(p['index'] for p in parents if p['group_key'] == cand.group
                         and all(t in opts for t, opts in zip(p['tokens'], cand.options)))
        if expected != cand.covered:
            raise AssertionError('Coverage verification failed.')
    return True


def save_results(tables, normalized_input, reports, solver_report, settings, output_dir, provenance,
                 raw_input=None, exclusions=None):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    for key, frame in tables.items():
        frame.to_csv(output_dir / f'{key}.csv', index=False)
    (raw_input if raw_input is not None else normalized_input).to_csv(output_dir / 'input_records.csv', index=False)
    normalized_input.to_csv(output_dir / 'normalized_sequences.csv', index=False)
    if exclusions is None:
        exclusions = pd.DataFrame(columns=['input_record', 'id', 'sequence', 'reason'])
    exclusions.to_csv(output_dir / 'excluded_sequences.csv', index=False)
    metadata = {'tool': 'Peptide Library Designer', 'version': '1.0.3', 'input': provenance,
                'input_validation': {'accepted_records': len(normalized_input),
                                     'excluded_records': len(exclusions),
                                     'unique_accepted_sequence_groups': len(tables['coverage'])},
                'settings': settings, 'candidate_generation': reports, 'solver': solver_report,
                'exhaustive_candidate_space': all(r['enumeration_complete'] for r in reports),
                'global_primary_optimum_certified': all(r['enumeration_complete'] for r in reports)
                                                   and solver_report['primary_optimal_over_candidates'],
                'environment': {'python': platform.python_version(), 'numpy': np.__version__,
                                'pandas': pd.__version__, 'scipy': scipy.__version__}}
    (output_dir / 'run_metadata.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')
    return metadata


def plot_results(tables, output_dir, title_prefix=''):
    design, coverage, composition = tables['design'], tables['coverage'], tables['composition']
    library_ids = design['library_id'].tolist()
    hit_matrix = np.array([[int(lid in json.loads(row.libraries_json)) for lid in library_ids]
                          for row in coverage.itertuples()])
    order = np.argsort([next((j for j,v in enumerate(row) if v), len(library_ids)) for row in hit_matrix], kind='stable')
    fig, axes = plt.subplots(1, 2, figsize=(11, max(5, min(13, len(coverage)*.18))),
                             gridspec_kw={'width_ratios': [1.4, 1]})
    axes[0].imshow(hit_matrix[order], aspect='auto', interpolation='nearest', cmap='Blues', vmin=0, vmax=1)
    axes[0].set_xticks(range(len(library_ids)), library_ids)
    axes[0].set_yticks(range(len(coverage)),
                      [json.loads(coverage.iloc[i]['parent_ids_json'])[0] for i in order], fontsize=7)
    axes[0].set_title('Starting-sequence coverage', loc='left', fontweight='bold')
    axes[0].set_xlabel('Selected library')
    bars = axes[1].bar(library_ids, design['size'], color='#137C8B')
    axes[1].bar_label(bars, padding=3)
    axes[1].set_ylim(0, max(design['size'])*1.17)
    axes[1].set_ylabel('Peptide combinations')
    axes[1].set_title('Library sizes', loc='left', fontweight='bold')
    axes[1].spines[['top','right']].set_visible(False)
    prefix = title_prefix + ' | ' if title_prefix else ''
    fig.suptitle(prefix + f"{int(coverage['covered'].sum())}/{len(coverage)} unique starting sequences covered",
                 fontsize=12, fontweight='bold')
    fig.tight_layout(rect=[0,0,1,.96])
    for suffix in ['png', 'svg']:
        fig.savefig(Path(output_dir) / f'coverage.{suffix}', dpi=170, bbox_inches='tight')
    plt.show()
    plt.close(fig)
    matrix = composition.pivot(index='library_id', columns='position', values='option_count').reindex(library_ids)
    fig, ax = plt.subplots(figsize=(max(8, min(15, len(matrix.columns)*.6)), max(3, len(matrix)*.6+1.5)))
    im = ax.imshow(matrix, aspect='auto', cmap='YlGnBu', vmin=1)
    ax.set_xticks(range(len(matrix.columns)), matrix.columns)
    ax.set_yticks(range(len(matrix)), matrix.index)
    for i in range(len(matrix)):
        for j in range(len(matrix.columns)):
            if pd.notna(matrix.iloc[i,j]):
                dark = matrix.iloc[i,j] > (1 + np.nanmax(matrix.to_numpy())) / 2
                ax.text(j, i, str(int(matrix.iloc[i,j])), ha='center', va='center', fontsize=9,
                        color='white' if dark else '#142B40')
    ax.set_xlabel('Sequence position')
    ax.set_title(prefix + 'Residue options per position (1 = fixed)', loc='left', fontweight='bold')
    fig.colorbar(im, ax=ax, label='Number of residue options')
    fig.tight_layout()
    for suffix in ['png', 'svg']:
        fig.savefig(Path(output_dir) / f'composition.{suffix}', dpi=170, bbox_inches='tight')
    plt.show()
    plt.close(fig)


def library_logo_frequencies(tables, token_mode='letters'):
    """Count residues over all enumerated members, with equal weight per member."""
    from collections import Counter
    records = []
    for design in tables['design'].itertuples(index=False):
        sequences = tables['members'].loc[tables['members']['library_id'] == design.library_id, 'sequence']
        tokens = [tokenize(sequence, token_mode) for sequence in sequences]
        if len(tokens) != int(design.size) or not tokens:
            raise ValueError(f'{design.library_id}: logo input does not contain all library members.')
        if len(set(tokens)) != len(tokens):
            raise ValueError(f'{design.library_id}: duplicate member sequences would bias the logo.')
        if any(len(sequence) != int(design.length) for sequence in tokens):
            raise ValueError(f'{design.library_id}: member lengths do not match the design.')
        for position in range(int(design.length)):
            counts = Counter(sequence[position] for sequence in tokens)
            for residue, count in sorted(counts.items()):
                records.append({'library_id': design.library_id, 'position': position+1,
                                'residue': residue, 'count': count, 'library_size': len(tokens),
                                'frequency': count / len(tokens)})
    return pd.DataFrame(records, columns=['library_id','position','residue','count','library_size','frequency'])


def logo_heights(frequencies, mode='probability', token_mode='letters'):
    """Probability heights, or frequency times information relative to 20 equiprobable amino acids."""
    if mode not in {'probability', 'information'}:
        raise ValueError('LOGO_MODE must be probability or information.')
    if mode == 'information' and token_mode != 'letters':
        raise ValueError('Information logos use the 20 canonical amino acids. Use probability for custom tokens.')
    result = frequencies.copy()
    result['height'] = result['frequency']
    if mode == 'information':
        for _, indices in result.groupby(['library_id','position']).groups.items():
            p = result.loc[indices, 'frequency'].to_numpy(dtype=float)
            information = max(0.0, math.log2(20) + float(np.sum(p * np.log2(p))))
            result.loc[indices, 'height'] = p * information
    return result


def plot_library_logos(tables, output_dir, token_mode='letters', mode='probability',
                       positions_per_row=30, title_prefix=''):
    """Display every library logo and save PNG, SVG, individual PDFs, and a multipage PDF."""
    from matplotlib.textpath import TextPath
    from matplotlib.font_manager import FontProperties
    from matplotlib.patches import PathPatch, Patch
    from matplotlib.transforms import Affine2D
    from matplotlib.backends.backend_pdf import PdfPages
    if not isinstance(positions_per_row, int) or isinstance(positions_per_row, bool) or positions_per_row < 1:
        raise ValueError('LOGO_POSITIONS_PER_ROW must be a positive integer.')
    if tables['design'].empty:
        raise ValueError('No selected libraries to plot.')
    frequencies = library_logo_frequencies(tables, token_mode)
    heights = logo_heights(frequencies, mode, token_mode)
    destination = Path(output_dir) / 'logos'
    destination.mkdir(parents=True, exist_ok=True)
    frequencies.to_csv(destination/'logo_frequencies.csv', index=False)
    heights.assign(scale=mode).to_csv(destination/'logo_heights.csv', index=False)
    groups = [('DE', '#C53B3F', 'D E'), ('KRH', '#2461A8', 'K R H'),
              ('STNQ', '#22865B', 'S T N Q'), ('AVILM', '#26384B', 'A V I L M'),
              ('FWY', '#8055A3', 'F W Y'), ('C', '#A36C00', 'C'), ('GP', '#C46D29', 'G P')]
    colors = {residue: color for residues, color, _ in groups for residue in residues}
    font = FontProperties(family='DejaVu Sans', weight='bold')
    glyphs = {str(residue): TextPath((0,0), str(residue), size=1, prop=font)
              for residue in frequencies['residue'].unique()}
    ymax = 1.0 if mode == 'probability' else math.log2(20)
    label = 'Frequency' if mode == 'probability' else 'Information (bits)'
    prefix = title_prefix + ' | ' if title_prefix else ''
    paths = []
    with PdfPages(destination/'all_library_logos.pdf') as combined:
        for design in tables['design'].itertuples(index=False):
            library_id, length = str(design.library_id), int(design.length)
            # Library IDs are generated internally, but prevent accidental unsafe output names.
            if not re.fullmatch(r'[A-Za-z0-9_-]+', library_id):
                raise ValueError('Library IDs for plot filenames must contain letters, digits, underscores, or hyphens.')
            nrows = math.ceil(length / positions_per_row)
            width = max(8.5, min(18.0, min(length, positions_per_row)*.43 + 1.8))
            fig, axes = plt.subplots(nrows, 1, figsize=(width, 2.4*nrows + 1.25), squeeze=False)
            data = heights.loc[heights['library_id'] == library_id]
            for row_number, ax in enumerate(axes.flat):
                start = row_number * positions_per_row + 1
                end = min(length, start + positions_per_row - 1)
                for position in range(start, end+1):
                    stack = data.loc[data['position'] == position].sort_values(['height','residue'])
                    bottom = 0.0
                    for entry in stack.itertuples(index=False):
                        height = float(entry.height)
                        if height <= 1e-12:
                            continue
                        glyph = glyphs[str(entry.residue)]
                        box = glyph.get_extents()
                        transform = (Affine2D().translate(-box.x0, -box.y0)
                                     .scale(.85/box.width, height/box.height)
                                     .translate(position-.425, bottom))
                        ax.add_patch(PathPatch(glyph, transform=transform+ax.transData,
                                               facecolor=colors.get(str(entry.residue), '#667583'), lw=0))
                        bottom += height
                ax.set_xlim(start-.6, start+min(length, positions_per_row)-.4)
                ax.set_ylim(0, ymax*1.02)
                ax.set_xticks(range(start, end+1))
                ax.set_yticks([0,.5,1] if mode == 'probability' else [0,1,2,3,4])
                ax.set_xlabel('Sequence position')
                ax.set_ylabel(label)
                ax.tick_params(axis='x', labelsize=9, length=3)
                ax.spines[['top','right']].set_visible(False)
                ax.spines['left'].set_color('#97A4B1')
                ax.spines['bottom'].set_color('#97A4B1')
                ax.spines['bottom'].set_bounds(start-.6, end+.6)
            title = f'{prefix}{library_id} | {int(design.size)} designed peptides'
            fig.suptitle(title, fontsize=13, fontweight='bold', y=.985)
            description = ('Letter height = fraction of designed library members; equal weight per member.'
                           if mode == 'probability' else
                           'Letter height = frequency × information; uniform 20-amino-acid background.')
            fig.text(.5, .09, description, ha='center', fontsize=9, color='#50647A')
            if token_mode == 'letters':
                handles = [Patch(facecolor=color, label=text) for _, color, text in groups]
                fig.legend(handles=handles, loc='lower center', ncol=7, frameon=False,
                           fontsize=8, bbox_to_anchor=(.5, .002))
            fig.tight_layout(rect=[0, .17 if token_mode == 'letters' else .12, 1, .935])
            for suffix in ['png','svg','pdf']:
                path = destination / f'{library_id}_logo.{suffix}'
                fig.savefig(path, dpi=220, bbox_inches='tight')
                paths.append(path)
            combined.savefig(fig, bbox_inches='tight')
            plt.show()
            plt.close(fig)
    print(f'Saved logos for {len(tables["design"])} libraries to {destination}; '
          f'scale={mode}. Combined PDF: all_library_logos.pdf')
    return frequencies, paths

7. Run the design
This cell reads the specified file, summarizes candidate generation, and distinguishes
optimality over generated candidates from exhaustive search. A missing file raises an
error; the notebook never substitutes example sequences automatically.

In [ ]:
raw_input = read_sequences(INPUT_PATH)
input_bytes = Path(INPUT_PATH).read_bytes()
provenance = {
    "source_name": Path(INPUT_PATH).name,
    "run_label": RUN_LABEL,
    "sha256": hashlib.sha256(input_bytes).hexdigest(),
    "records": len(raw_input),
}
parents, normalized_input, exclusions = prepare_input(
    raw_input, SEQUENCE_COLUMN, ID_COLUMN, WEIGHT_COLUMN, GROUP_COLUMN, TOKEN_MODE,
    INVALID_SEQUENCE_POLICY)
print(f"Input: {Path(INPUT_PATH).name} | {RUN_LABEL}")
print(f"{len(raw_input)} input records; {len(normalized_input)} accepted records; "
      f"{len(exclusions)} excluded; {len(parents)} unique accepted sequence/group entries.")
if not exclusions.empty:
    print("Excluded input sequences:")
    print(exclusions.to_string(index=False))
started = time.perf_counter()
candidates, candidate_report = build_candidates(
    parents, MAX_LIBRARY_SIZE, CANDIDATE_MODE, MAX_CANDIDATES_PER_GROUP,
    MAX_ENUMERATION_TRANSITIONS, GREEDY_RESTARTS, RANDOM_SEED)
print(pd.DataFrame(candidate_report).to_string(index=False))
selected, solver_report = optimize_libraries(parents, candidates, MAX_LIBRARIES, SOLVER_TIME_LIMIT)
tables = result_tables(parents, selected, TOKEN_MODE)
verify_design(parents, selected, tables, MAX_LIBRARIES, MAX_LIBRARY_SIZE)
settings = {k: globals()[k] for k in [
    "RUN_LABEL", "SEQUENCE_COLUMN", "ID_COLUMN", "WEIGHT_COLUMN", "GROUP_COLUMN", "TOKEN_MODE",
    "INVALID_SEQUENCE_POLICY",
    "MAX_LIBRARIES", "MAX_LIBRARY_SIZE", "CANDIDATE_MODE", "MAX_CANDIDATES_PER_GROUP",
    "MAX_ENUMERATION_TRANSITIONS", "GREEDY_RESTARTS", "RANDOM_SEED", "SOLVER_TIME_LIMIT",
    "LOGO_MODE", "LOGO_POSITIONS_PER_ROW"]}
metadata = save_results(tables, normalized_input, candidate_report, solver_report,
                        settings, OUTPUT_DIR, provenance, raw_input, exclusions)
print(f"Selected {len(selected)} libraries; covered "
      f"{solver_report['covered_unique_parents']}/{len(parents)} unique starting sequences.")
print(f"Summed library members: {solver_report['summed_library_members']}")
print("Primary optimum certified over candidate pool:", solver_report["primary_optimal_over_candidates"])
print("Exhaustive candidate space:", metadata["exhaustive_candidate_space"])
print("Global primary optimum certified:", metadata["global_primary_optimum_certified"])
print("Secondary cost-solve status:", solver_report["cost_status"])
print(f"Completed in {time.perf_counter()-started:.1f} seconds. Outputs: {OUTPUT_DIR}")
print(tables["design"][["library_id", "group", "length", "size", "unique_parents_covered"]].to_string(index=False))

8. Figures
Blue cells show exact inclusion of an input sequence. A value of 1 in the composition
figure denotes a position fixed within that library. Figures are saved as SVG and PNG.
Inspect coverage.csv for inputs that could not be included within the chosen budget.

Each selected library is displayed as a WebLogo-style stack of amino-acid letters.
The default LOGO_MODE = "probability" uses letter heights equal to residue frequency
across all enumerated members of that library, with one vote per member. Each
column sums to 1. These are designed sequence proportions, not measured synthesis
abundances or the frequencies among the starting hits.

Set LOGO_MODE = "information" for heights in bits relative to a uniform background
of 20 canonical amino acids: stack height = log₂(20) − entropy. No finite-sample
correction is applied because the complete designed library is enumerated.
Information mode requires canonical letter sequences; probability mode also supports
explicit monomer labels. Long sequences wrap at LOGO_POSITIONS_PER_ROW.

Each logo is saved under OUTPUT_DIR/logos/ as PNG, SVG, and PDF.
all_library_logos.pdf contains one page per library. The exact frequencies and
plotted heights are saved as CSV tables. All plotting code is included here;
no WebLogo server, external program, or additional Python package is required.

In [ ]:
plot_results(tables, OUTPUT_DIR, title_prefix=RUN_LABEL)

logo_frequencies, logo_paths = plot_library_logos(
    tables, OUTPUT_DIR, token_mode=TOKEN_MODE, mode=LOGO_MODE,
    positions_per_row=LOGO_POSITIONS_PER_ROW, title_prefix=RUN_LABEL)

print("First library members:")
print(tables["members"].head(12).to_string(index=False))
print("\nUncovered starting sequences:")
print(tables["coverage"].loc[~tables["coverage"]["covered"],
      ["parent_ids_json", "sequence", "weight"]].to_string(index=False))